# 04 Periods and dimensions
Three facts about the raw data decide whether your sums mean anything: dates come in several formats, a 10-Q carries **two** schedules (this period and the prior fiscal year-end), and one holding can occupy **several** rows.

In [ ]:
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "params.yaml").exists())
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd
from lookthrough import paths, acquire, load, mapping, holdings, clean, reconcile, report, tables, pipeline
P = paths.load_params(ROOT)
RDIR = paths.release_dir(P, ROOT)
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 20)
print("release folder:", RDIR.relative_to(ROOT), "| fixture mode:", P["data"]["use_fixture"])

In [ ]:
raw = pd.read_csv(RDIR / "soi.tsv", sep="\t", dtype=str, nrows=3, usecols=["ddate", "period"])
print("Raw SOI dates:", raw.iloc[0].to_dict())
print("Normalized:   ", load.norm_date(raw["ddate"]).iloc[0], load.norm_date(raw["period"]).iloc[0])

In [ ]:
sub, tag = load.load_sub(RDIR), load.load_tag(RDIR)
header = load.soi_header(RDIR)
colmap = mapping.map_soi_columns(header, tag)
soi = load.load_soi(RDIR, usecols=mapping.soi_usecols(header, colmap))   # only the columns we use
soi = soi[soi["adsh"].isin(sub["adsh"])]                                  # drop superseded filings
canon = holdings.split_periods(mapping.to_canonical(soi, colmap), sub)  # current period from SUB
print("Rows per filing, current vs comparative (current = ddate equals the filing's period in SUB):")
print(canon.groupby(["name", "is_current"]).size().unstack(fill_value=0).rename(columns={True: "current", False: "comparative"}))

In [ ]:
num = load.load_num(RDIR, tags=[reconcile.FV_TAG])
for s in num[num["dimn"] != "0"]["segments"].head(4):
    print(mapping.parse_segments(s))

In [ ]:
h = holdings.coalesce_holdings(canon)
cur = h[h["is_current"]]
print(f"Current-period SOI rows: {int(canon['is_current'].sum())} -> holdings after coalescing: {len(cur)}")
print(f"Current rows without an identifier (subtotals, or holdings on a custom axis): {int((canon['is_current'] & canon['identifier'].isna()).sum())}")
print("Rows merged per holding:", cur["n_rows"].value_counts().sort_index().to_dict())
print("Holdings whose merged rows disagree on a value:")
print(cur[cur["n_conflicts"] > 0][["name", "identifier", "n_rows", "n_conflicts"]].to_string(index=False))

**Your turn.** For every holding with conflicts, decide which value is right by checking the filing, and write the rule you apply (and why) in your cleaning log.